# AdaptiveDepth Phase 5 — Google Drive evaluation
This notebook mounts Google Drive, copies the complete `checkpoint-step-000500` folder to Colab local storage, runs the Phase 5 post-training gate, and saves the result archive back to Drive. Use a T4 GPU runtime.

In [ ]:
import base64, os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nFor a private repository, add a read-only GITHUB_TOKEN to Colab Secrets.')
os.chdir(repo)
print('Repository:', repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before evaluating Phase 5'
print('GPU:', torch.cuda.get_device_name(0))

## Mount Drive and copy the checkpoint
Upload the entire `checkpoint-step-000500` folder to **My Drive** first. If you put it inside another Drive folder, edit `DRIVE_CHECKPOINT` below. The copy to `/content` is intentional: loading a 1.9 GB model directly from mounted Drive is slower and less reliable.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
DRIVE_CHECKPOINT = pathlib.Path('/content/drive/MyDrive/checkpoint-step-000500')
LOCAL_CHECKPOINT = pathlib.Path('/content/checkpoint-step-000500')

required = ['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']
missing = [name for name in required if not (DRIVE_CHECKPOINT / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'Checkpoint not found at {DRIVE_CHECKPOINT} or files are missing: {missing}. '
        'Edit DRIVE_CHECKPOINT to match the folder location in My Drive.'
    )
model_size_gb = (DRIVE_CHECKPOINT / 'model.safetensors').stat().st_size / 1024**3
if model_size_gb < 1.5:
    raise RuntimeError(f'model.safetensors is unexpectedly small: {model_size_gb:.3f} GB')

if LOCAL_CHECKPOINT.exists():
    shutil.rmtree(LOCAL_CHECKPOINT)
print(f'Copying {model_size_gb:.2f} GB checkpoint from Drive to local disk...')
shutil.copytree(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
local_size_gb = (LOCAL_CHECKPOINT / 'model.safetensors').stat().st_size / 1024**3
if local_size_gb != model_size_gb:
    raise RuntimeError('The local checkpoint copy is incomplete')
checkpoint = LOCAL_CHECKPOINT
print('Local checkpoint ready:', checkpoint)
print('Model size GB:', round(local_size_gb, 3))

In [ ]:
workspace = pathlib.Path('experiments/phase5_evaluation')
command = [
    'adaptivedepth', 'run-phase5-evaluation',
    '--checkpoint', str(checkpoint),
    '--config', 'configs/cloud_t4.yaml',
    '--routes-file', 'configs/phase5_routes.yaml',
    '--dataset', 'data/phase4_eval.jsonl',
    '--reference-file', 'data/phase4_reference_metrics.json',
    '--evaluation-config', 'configs/phase5_evaluation.yaml',
    '--workspace', str(workspace),
    '--resume',
]
subprocess.run(command, check=True)

In [ ]:
print((workspace / 'gate.md').read_text())
archive = pathlib.Path('/content/phase5_evaluation_artifacts.zip')
subprocess.run([sys.executable, '-m', 'zipfile', '-c', str(archive), str(workspace)], check=True)
drive_results = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
drive_results.mkdir(parents=True, exist_ok=True)
saved_archive = drive_results / archive.name
shutil.copy2(archive, saved_archive)
shutil.copy2(workspace / 'gate.md', drive_results / 'phase5_gate.md')
shutil.copy2(workspace / 'gate.json', drive_results / 'phase5_gate.json')
print('Saved evaluation archive to:', saved_archive)
print('Upload the archive or paste the gate output here for analysis.')